# uGMRT Preprocess Workflow (Production)

This notebook is a thin workflow wrapper around module functions in `ugmrt_query.py`.

Core loop:
1. Derive bandpass
2. Run diagnostics
3. Propose/write flag table updates
4. Repeat from Step 1 with one or more flag tables

All operations are non-destructive: input visibilities are never modified on disk.

In [ ]:
import importlib
import sys
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

print('ugmrt_query loaded')

In [ ]:
# User configuration
ITER_TAG = 'iter03'

BASE_DIR = Path('/media/wasim/gmrt_40_014')
WORK_DIR = BASE_DIR / 'work'

CAL_FITS = WORK_DIR / '40_014_25JUL2021_copy' / '40_014_25jul2021_gsb.FITS'
INDEX_CACHE = WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'

# Row-index cache validation mode:
#   'fast'      -> compare file size + modification time
#   'fast+sha'  -> fast check first; on mismatch, verify SHA before rebuild
#   'sha256'    -> compare full SHA256 hash every run (strict, slower)
#   'none'      -> trust cache without checking source file
INDEX_VALIDATION_MODE = 'fast+sha'

# Multiple on-disk flag tables can be merged on-the-fly
FLAG_TABLE_BASE = WORK_DIR / '3c48_flag_table.json'
FLAG_TABLE_SESSION = WORK_DIR / '3c48_flag_table_session.json'
FLAG_TABLE_PATHS = [p for p in [FLAG_TABLE_BASE, FLAG_TABLE_SESSION] if p.exists()]

# In-memory flag tables proposed in earlier dry-run iterations can also be reused on-the-fly.
if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

# User control: set USE_PENDING_FLAG_TABLES=False if you do not want dry-run proposals
# from previous iterations to affect the next bandpass solve.
USE_PENDING_FLAG_TABLES = True

# Optional reset switch: set CLEAR_PENDING_FLAG_TABLES=True once to discard any previously
# carried in-memory proposals, then run this cell.
CLEAR_PENDING_FLAG_TABLES = False
if CLEAR_PENDING_FLAG_TABLES:
    PENDING_FLAG_TABLES = []

DRY_RUN_BANDPASS = True
DRY_RUN_FLAG_WRITE = True

# Solve options
SOURCE = '3C48'
STOKES = ('RR', 'LL')
CHAN_RANGE = (64, 191)
MAX_ROWS_SOLVE = 150_000
SMOOTH_WINDOW = 5
MIN_BASELINES = 20
COUPLE_STOKES_FLAGS = True

# Diagnostics options
EXCLUDE_FOR_PLOTS = []
SKIP_EDGE_CHANNELS = (0, 0)
TOP_N = 12
DIAG_APPLY_FLAGS_ON_THE_FLY = True
DIAG_SAVE_UNFLAGGED_COMPARISON = False

print('CAL_FITS:', CAL_FITS)
print('INDEX_CACHE:', INDEX_CACHE)
print('INDEX_VALIDATION_MODE:', INDEX_VALIDATION_MODE)
print('FLAG_TABLE_PATHS:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('USE_PENDING_FLAG_TABLES:', USE_PENDING_FLAG_TABLES)
print('CLEAR_PENDING_FLAG_TABLES:', CLEAR_PENDING_FLAG_TABLES)
print('ITER_TAG:', ITER_TAG)
print('DRY_RUN_BANDPASS:', DRY_RUN_BANDPASS)
print('DRY_RUN_FLAG_WRITE:', DRY_RUN_FLAG_WRITE)
print('COUPLE_STOKES_FLAGS:', COUPLE_STOKES_FLAGS)
print('DIAG_APPLY_FLAGS_ON_THE_FLY:', DIAG_APPLY_FLAGS_ON_THE_FLY)
print('DIAG_SAVE_UNFLAGGED_COMPARISON:', DIAG_SAVE_UNFLAGGED_COMPARISON)

In [ ]:
# Step 1: Load or build persistent row index cache
row_index = q.get_or_build_row_index(
    CAL_FITS,
    cache_path=INDEX_CACHE,
    force_rebuild=False,
    validation_mode=INDEX_VALIDATION_MODE,
    write_cache=True,
 )

print('Index path:', row_index.get('index_cache_path', INDEX_CACHE))
print('Source identity:', row_index.get('source_identity'))
print('Source SHA256:', row_index.get('source_sha256'))

In [ ]:
# Step 2: Derive bandpass (non-destructive)
BANDPASS_OUT_BASE = WORK_DIR / '3c48_bandpass_25jul_gsb.npz'
active_pending_flag_tables = PENDING_FLAG_TABLES if USE_PENDING_FLAG_TABLES else []

bandpass_run = q.derive_bandpass_iteration(
    fits_path=CAL_FITS,
    index=row_index,
    bandpass_out=BANDPASS_OUT_BASE,
    source=SOURCE,
    stokes=STOKES,
    chan_range=CHAN_RANGE,
    max_rows=MAX_ROWS_SOLVE,
    smooth_window=SMOOTH_WINDOW,
    min_baselines=MIN_BASELINES,
    ignore_autos=True,
    flag_table_path=FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table=active_pending_flag_tables if active_pending_flag_tables else None,
    couple_stokes_flags=COUPLE_STOKES_FLAGS,
    iteration_tag=ITER_TAG,
    dry_run=DRY_RUN_BANDPASS,
 )

bandpass_sol = bandpass_run['solution']
bandpass_out_path = bandpass_run['bandpass_out']

print('Bandpass dry-run:', bandpass_run['dry_run'])
print('Bandpass output path:', bandpass_out_path)
print('On-disk flag tables used:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('Pending in-memory flag tables applied:', len(active_pending_flag_tables))
print('Merged flag tables seen by solve:', bandpass_sol.get('flag_table_paths', []))
print('Merged flag table count:', bandpass_sol.get('flag_table_count', 0))
print('Rows dropped by merged flags:', bandpass_sol.get('solve_dropped_rows_by_flag_table', 0))

In [ ]:
# Step 3: Diagnostics + Step 4: Propose flags + Step 5: Update flag table
DIAG_PLOT_BASE = WORK_DIR / '3c48_bandpass_diagnostics.png'
diag_plot_path = q.tagged_output_path(DIAG_PLOT_BASE, ITER_TAG)

# Use the same merged table inputs as the solve stage, but make diagnostics application optional.
diag_flag_table_paths = FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None
diag_flag_tables = active_pending_flag_tables if active_pending_flag_tables else None

diag = q.run_bandpass_diagnostics(
    row_index,
    bandpass_sol,
    source=SOURCE,
    chan_range=CHAN_RANGE,
    stokes=STOKES,
    max_rows=60_000,
    exclude_antennas=EXCLUDE_FOR_PLOTS,
    apply_flag_tables=DIAG_APPLY_FLAGS_ON_THE_FLY,
    flag_table_path=diag_flag_table_paths,
    flag_table=diag_flag_tables,
    strict_flag_table=False,
    couple_stokes_flags=COUPLE_STOKES_FLAGS,
    skip_edge_channels=SKIP_EDGE_CHANNELS,
    top_n=TOP_N,
    title=f'3C48 diagnostics | {ITER_TAG}',
    save_path=diag_plot_path,
 )

print('Diagnostics apply merged flags:', diag['diagnostics_flag_table_applied'])
print('Diagnostics merged flag table count:', diag['diagnostics_flag_table_count'])
print('Diagnostics rows dropped by merged flags:', diag['diagnostics_dropped_rows_by_flag_table'])

if DIAG_SAVE_UNFLAGGED_COMPARISON and DIAG_APPLY_FLAGS_ON_THE_FLY:
    DIAG_PLOT_UNFLAGGED_BASE = WORK_DIR / '3c48_bandpass_diagnostics_unflagged.png'
    diag_plot_unflagged_path = q.tagged_output_path(DIAG_PLOT_UNFLAGGED_BASE, ITER_TAG)
    q.run_bandpass_diagnostics(
        row_index,
        bandpass_sol,
        source=SOURCE,
        chan_range=CHAN_RANGE,
        stokes=STOKES,
        max_rows=60_000,
        exclude_antennas=EXCLUDE_FOR_PLOTS,
        apply_flag_tables=False,
        couple_stokes_flags=COUPLE_STOKES_FLAGS,
        skip_edge_channels=SKIP_EDGE_CHANNELS,
        top_n=TOP_N,
        title=f'3C48 diagnostics (unflagged) | {ITER_TAG}',
        save_path=diag_plot_unflagged_path,
     )
    print('Saved unflagged comparison plot:', diag_plot_unflagged_path)

proposal = q.propose_flag_updates_from_diagnostics(
    diag,
    pol='LL',
    mode='both',
    antenna_flag_threshold_jy=180.0,
    baseline_flag_threshold_jy=800.0,
    max_antennas_to_flag=1,
    max_baselines_to_flag=6,
 )

print('Candidate antennas:', proposal['candidate_antennas'])
print('Candidate baselines:', proposal['candidate_baselines'])

# Build the next merged table from all available sources so dry-run propagation stays cumulative.
flag_update = q.update_flag_table(
    output_path=FLAG_TABLE_SESSION,
    add_antennas=proposal['proposal']['bad_antennas'],
    add_baselines=proposal['proposal']['bad_baselines'],
    base_flag_tables=active_pending_flag_tables if active_pending_flag_tables else None,
    base_flag_table_paths=FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    notes=f'Auto-proposed from diagnostics {ITER_TAG}',
    dry_run=DRY_RUN_FLAG_WRITE,
 )

# Even in dry-run mode, keep the merged proposed table in memory so the next iteration
# can use it on-the-fly without writing anything to disk.
if DRY_RUN_FLAG_WRITE:
    PENDING_FLAG_TABLES = [flag_update['flag_table']]
else:
    PENDING_FLAG_TABLES = []

print('Flag update dry-run:', flag_update['dry_run'])
print('Flag table target:', flag_update['output_path'])
print('Would add antennas:', flag_update['added_antennas'])
print('Would add baselines:', flag_update['added_baselines'])
print('Merged cumulative table antennas:', len(flag_update['flag_table'].get('bad_antennas', [])))
print('Merged cumulative table baselines:', len(flag_update['flag_table'].get('bad_baselines', [])))
print('Pending in-memory flag tables for next iteration:', len(PENDING_FLAG_TABLES))

In [ ]:
# Optional automation: run multiple iterations with auto-incremented tags.
# This cell is self-contained: it does not depend on the earlier config/index cells.
import importlib
import sys
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

AUTO_BASE_DIR = Path('/media/wasim/gmrt_40_014')
AUTO_WORK_DIR = AUTO_BASE_DIR / 'work'
AUTO_CAL_FITS = AUTO_WORK_DIR / '40_014_25JUL2021_copy' / '40_014_25jul2021_gsb.FITS'
AUTO_INDEX_CACHE = AUTO_WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'
AUTO_INDEX_VALIDATION_MODE = 'fast+sha'

AUTO_FLAG_TABLE_BASE = AUTO_WORK_DIR / '3c48_flag_table.json'
AUTO_FLAG_TABLE_SESSION = AUTO_WORK_DIR / '3c48_flag_table_session.json'
AUTO_FLAG_TABLE_PATHS = [p for p in [AUTO_FLAG_TABLE_BASE, AUTO_FLAG_TABLE_SESSION] if p.exists()]

if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

AUTO_PENDING_FLAG_TABLES = list(PENDING_FLAG_TABLES)
AUTO_USE_PENDING_FLAG_TABLES = True
AUTO_CLEAR_PENDING_FLAG_TABLES = False
if AUTO_CLEAR_PENDING_FLAG_TABLES:
    AUTO_PENDING_FLAG_TABLES = []

AUTO_START_ITER = 5
AUTO_N_ITERS = 4
AUTO_ITER_PREFIX = 'iter'
AUTO_ITERATION_WIDTH = 2

AUTO_DRY_RUN_BANDPASS = True
AUTO_DRY_RUN_FLAG_WRITE = True

AUTO_SOURCE = '3C48'
AUTO_STOKES = ('RR', 'LL')
AUTO_CHAN_RANGE = (64, 191)
AUTO_MAX_ROWS_SOLVE = 150_000
AUTO_SMOOTH_WINDOW = 5
AUTO_MIN_BASELINES = 20
AUTO_COUPLE_STOKES_FLAGS = True

AUTO_EXCLUDE_FOR_PLOTS = []
AUTO_SKIP_EDGE_CHANNELS = (0, 0)
AUTO_TOP_N = 12
AUTO_MAX_ROWS_DIAG = 60_000
AUTO_DIAG_APPLY_FLAGS_ON_THE_FLY = True
AUTO_DIAG_SAVE_UNFLAGGED_COMPARISON = False

AUTO_BANDPASS_OUT_BASE = AUTO_WORK_DIR / '3c48_bandpass_25jul_gsb.npz'
AUTO_DIAG_PLOT_BASE = AUTO_WORK_DIR / '3c48_bandpass_diagnostics.png'
AUTO_DIAG_UNFLAGGED_PLOT_BASE = AUTO_WORK_DIR / '3c48_bandpass_diagnostics_unflagged.png'

auto_result = q.run_iterative_bandpass_workflow(
    fits_path=AUTO_CAL_FITS,
    index_cache_path=AUTO_INDEX_CACHE,
    index_validation_mode=AUTO_INDEX_VALIDATION_MODE,
    write_index_cache=True,
    bandpass_out_base=AUTO_BANDPASS_OUT_BASE,
    diag_plot_base=AUTO_DIAG_PLOT_BASE,
    diag_plot_unflagged_base=AUTO_DIAG_UNFLAGGED_PLOT_BASE,
    flag_table_session_path=AUTO_FLAG_TABLE_SESSION,
    base_flag_table_paths=AUTO_FLAG_TABLE_PATHS,
    pending_flag_tables=AUTO_PENDING_FLAG_TABLES,
    use_pending_flag_tables=AUTO_USE_PENDING_FLAG_TABLES,
    start_iteration=AUTO_START_ITER,
    n_iterations=AUTO_N_ITERS,
    iter_prefix=AUTO_ITER_PREFIX,
    iter_width=AUTO_ITERATION_WIDTH,
    dry_run_bandpass=AUTO_DRY_RUN_BANDPASS,
    dry_run_flag_write=AUTO_DRY_RUN_FLAG_WRITE,
    source=AUTO_SOURCE,
    stokes=AUTO_STOKES,
    chan_range=AUTO_CHAN_RANGE,
    max_rows_solve=AUTO_MAX_ROWS_SOLVE,
    smooth_window=AUTO_SMOOTH_WINDOW,
    min_baselines=AUTO_MIN_BASELINES,
    max_rows_diag=AUTO_MAX_ROWS_DIAG,
    exclude_for_plots=AUTO_EXCLUDE_FOR_PLOTS,
    diag_apply_flag_tables=AUTO_DIAG_APPLY_FLAGS_ON_THE_FLY,
    diag_save_unflagged_comparison=AUTO_DIAG_SAVE_UNFLAGGED_COMPARISON,
    couple_stokes_flags=AUTO_COUPLE_STOKES_FLAGS,
    skip_edge_channels=AUTO_SKIP_EDGE_CHANNELS,
    top_n=AUTO_TOP_N,
    proposal_pol='LL',
    proposal_mode='both',
    antenna_flag_threshold_jy=180.0,
    baseline_flag_threshold_jy=800.0,
    max_antennas_to_flag=1,
    max_baselines_to_flag=6,
    strict_flag_table=False,
)

# Synchronize useful notebook globals with the latest automation result.
PENDING_FLAG_TABLES = auto_result['pending_flag_tables']
row_index = auto_result['index']
bandpass_run = auto_result['last_bandpass_run']
bandpass_sol = bandpass_run['solution'] if bandpass_run is not None else None
diag = auto_result['last_diagnostics']
flag_update = auto_result['last_flag_update']

print('Auto iterations executed:', len(auto_result['history']))
print('AUTO_ITERATION_WIDTH:', AUTO_ITERATION_WIDTH, '-> tags like', f"{AUTO_ITER_PREFIX}{AUTO_START_ITER:0{AUTO_ITERATION_WIDTH}d}")
print('AUTO_COUPLE_STOKES_FLAGS:', AUTO_COUPLE_STOKES_FLAGS)
for h in auto_result['history']:
    print(
        f"{h['iteration_tag']}: "
        f"cand_ant={len(h['candidate_antennas'])}, "
        f"cand_base={len(h['candidate_baselines'])}, "
        f"cum_ant={h['cumulative_bad_antenna_count']}, "
        f"cum_base={h['cumulative_bad_baseline_count']}, "
        f"diag_drop={h['diagnostics_dropped_rows_by_flag_table']}, "
        f"plot={h['diagnostics_plot_path']}"
    )
print('Pending in-memory flag tables now:', len(PENDING_FLAG_TABLES))

## Iteration Loop
For the next iteration:
1. Set `ITER_TAG` to the next value (for example `iter02`).
2. If you want outputs written, set `DRY_RUN_BANDPASS=False` and `DRY_RUN_FLAG_WRITE=False`.
3. Re-run cells 4 to 6.
4. On-disk flag tables are always taken from `FLAG_TABLE_PATHS`.
5. Dry-run proposals are kept in memory as `PENDING_FLAG_TABLES`, but they are only applied if `USE_PENDING_FLAG_TABLES=True`.
6. If you want to stop propagation completely, set `USE_PENDING_FLAG_TABLES=False`.
7. If you want to discard already-carried in-memory proposals, set `CLEAR_PENDING_FLAG_TABLES=True` once and rerun the configuration cell.
8. Set `DIAG_APPLY_FLAGS_ON_THE_FLY=True` to apply merged flag tables before diagnostics statistics.
9. Set `DIAG_SAVE_UNFLAGGED_COMPARISON=True` to save an additional unflagged diagnostic plot for side-by-side comparison.
10. The automation cell is self-contained and can run without first executing the manual config/index cells.
11. `AUTO_ITERATION_WIDTH` controls zero-padding of iteration tags: width `2` gives `iter01`, width `3` gives `iter001`.

All filtering and calibration are performed in memory; raw vis data on disk remains untouched.